# Модуль 2. Урок 4: Чистый SQL через CTE (Common Table Expressions)

Вместо вложенных подзапросов в 5 этажей, пишите читаемый код с `WITH`:


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(r"d:\vibe_app\main_data_analytics\course\datasets\ecommerce.db")

query = """
WITH customer_spending AS (
    SELECT 
        customer_id,
        ROUND(SUM(total_amount), 2) AS ltv
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'Completed'
    GROUP BY customer_id
),
thresholds AS (
    SELECT AVG(ltv) AS avg_spend FROM customer_spending
)
SELECT 
    cs.customer_id,
    cs.ltv,
    ROUND(t.avg_spend, 2) AS market_avg,
    CASE 
        WHEN cs.ltv >= t.avg_spend * 2 THEN 'VIP Tier'
        WHEN cs.ltv >= t.avg_spend THEN 'Above Average'
        ELSE 'Regular'
    END AS customer_tier
FROM customer_spending cs
CROSS JOIN thresholds t
ORDER BY cs.ltv DESC
LIMIT 10;
"""
print(pd.read_sql(query, conn))
